# 05 — Final Load Prep

Compute all KPIs and export the Tableau-ready datasets used in the dashboard.

**Exports produced:**
| File | Used for |
|---|---|
| `tableau_segment_kpi.csv` | KPI cards + segment comparison bar charts |
| `tableau_monthly_revenue.csv` | Revenue trend line chart |
| `tableau_country_summary.csv` | Geographic map view |
| `tableau_customer_detail.csv` | Scatter plots + customer-level filters |
| `tableau_transactions_with_segments.csv` | Drill-down transaction table |

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = PROJECT_ROOT / 'data' / 'processed'

rfm   = pd.read_csv(PROC / 'rfm_customer_segments.csv')
trans = pd.read_csv(
    PROC / 'online_retail_cleaned.csv',
    parse_dates=['invoice_date']
)
print(f"RFM customers : {len(rfm):,}")
print(f"Transactions  : {len(trans):,}")
print(f"Date range    : {trans['invoice_date'].min().date()} → {trans['invoice_date'].max().date()}")

RFM customers : 5,861
Transactions  : 776,641
Date range    : 2009-12-01 → 2011-12-09


## 1. Enrich Customer Records with Country

In [2]:
# Each customer's primary country = mode of their transaction countries
customer_country = (
    trans.groupby('customer_id')['country']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'country': 'primary_country'})
)
rfm_full = rfm.merge(customer_country, on='customer_id', how='left')
print(f"Customers with country: {rfm_full['primary_country'].notna().sum():,} / {len(rfm_full):,}")
rfm_full.head()

Customers with country: 5,861 / 5,861


,customer_id,recency,frequency,monetary,R_score,F_score,M_score,segment,primary_country
0,12346,325,12,77556.46,2,5,5,At Risk,UNITED KINGDOM
1,12347,1,8,4921.53,5,4,5,Champions,ICELAND
2,12348,74,5,1658.40,3,4,4,Big Spenders,FINLAND
3,12349,18,3,3678.69,5,3,5,Big Spenders,ITALY
4,12350,309,1,294.40,2,1,2,Lost,NORWAY


## 2. Segment KPI Summary

In [3]:
total_rev  = rfm['monetary'].sum()
total_cust = len(rfm)

segment_kpi = rfm.groupby('segment').agg(
    customers=('customer_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    median_monetary=('monetary', 'median'),
    min_monetary=('monetary', 'min'),
    max_monetary=('monetary', 'max')
).round(2).reset_index()

segment_kpi['pct_customers'] = (segment_kpi['customers'] / total_cust * 100).round(1)
segment_kpi['pct_revenue']   = (segment_kpi['total_revenue'] / total_rev * 100).round(1)

out = PROC / 'tableau_segment_kpi.csv'
segment_kpi.to_csv(out, index=False)
print(f"Saved: {out.name}")
display(segment_kpi[['segment','customers','pct_customers','total_revenue','pct_revenue',
                      'avg_recency','avg_frequency','avg_monetary']])

Saved: tableau_segment_kpi.csv


,segment,customers,pct_customers,total_revenue,pct_revenue,avg_recency,avg_frequency,avg_monetary
0,At Risk,1530,26.1,1928758.50,11.3,400.75,3.39,1260.63
1,Big Spenders,683,11.7,2370959.86,13.9,96.91,6.58,3471.39
2,Champions,1477,25.2,11822980.86,69.2,19.38,15.56,8004.73
3,Lost,788,13.4,217628.03,1.3,474.40,1.00,276.18
4,Loyal Customers,614,10.5,421945.80,2.5,67.60,3.46,687.21
5,Potential Loyalists,769,13.1,310805.24,1.8,63.13,1.39,404.17


## 3. Monthly Revenue Trend

In [4]:
trans['year_month'] = trans['invoice_date'].dt.to_period('M').astype(str)

monthly = (
    trans.groupby('year_month')
    .agg(
        total_revenue=('revenue', 'sum'),
        order_count=('invoice_no', 'nunique'),
        customer_count=('customer_id', 'nunique')
    )
    .round(2)
    .reset_index()
    .sort_values('year_month')
)
monthly['avg_order_value'] = (monthly['total_revenue'] / monthly['order_count']).round(2)
monthly['month_number']    = range(1, len(monthly) + 1)

out = PROC / 'tableau_monthly_revenue.csv'
monthly.to_csv(out, index=False)
print(f"Saved: {out.name}  ({len(monthly)} months)")
display(monthly.tail(6))

Saved: tableau_monthly_revenue.csv  (25 months)


,year_month,total_revenue,order_count,customer_count,avg_order_value,month_number
19,2011-07,591603.79,1321,946,447.85,20
20,2011-08,635514.38,1267,933,501.59,21
21,2011-09,938752.63,1739,1259,539.82,22
22,2011-10,1002326.56,1903,1361,526.71,23
23,2011-11,1136534.00,2642,1660,430.18,24
24,2011-12,512228.08,776,614,660.09,25


## 4. Country Summary (Geographic Map)

In [5]:
country_summary = (
    trans.groupby('country')
    .agg(
        total_revenue=('revenue', 'sum'),
        order_count=('invoice_no', 'nunique'),
        customer_count=('customer_id', 'nunique'),
        transaction_count=('invoice_no', 'count')
    )
    .round(2)
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)
country_summary['avg_order_value'] = (
    country_summary['total_revenue'] / country_summary['order_count']
).round(2)
country_summary['revenue_rank'] = country_summary['total_revenue'].rank(
    ascending=False).astype(int)

out = PROC / 'tableau_country_summary.csv'
country_summary.to_csv(out, index=False)
print(f"Saved: {out.name}  ({len(country_summary)} countries)")
display(country_summary.head(10))

Saved: tableau_country_summary.csv  (41 countries)


,country,total_revenue,order_count,customer_count,transaction_count,avg_order_value,revenue_rank
37,UNITED KINGDOM,14291209.57,33384,5336,699649,428.09,1
10,EIRE,588022.43,539,5,15360,1090.95,2
24,NETHERLANDS,549773.41,216,22,4981,2545.25,3
14,GERMANY,383419.24,756,107,15791,507.17,4
13,FRANCE,309459.89,591,94,13033,523.62,5
0,AUSTRALIA,167800.01,89,15,1783,1885.39,6
32,SPAIN,97994.50,147,41,3563,666.63,7
34,SWITZERLAND,93400.94,82,22,2951,1139.04,8
33,SWEDEN,86045.14,98,19,1267,878.01,9
9,DENMARK,67422.69,42,12,756,1605.30,10


## 5. Customer Detail with Segment & Country

In [6]:
cust_stats = (
    trans.groupby('customer_id')
    .agg(
        total_transactions=('invoice_no', 'count'),
        unique_invoices=('invoice_no', 'nunique'),
        unique_products=('stock_code', 'nunique'),
        first_purchase=('invoice_date', 'min'),
        last_purchase=('invoice_date', 'max')
    )
    .reset_index()
)

customer_detail = rfm_full.merge(cust_stats, on='customer_id', how='left')
customer_detail['customer_lifespan_days'] = (
    (customer_detail['last_purchase'] - customer_detail['first_purchase'])
    .dt.days
)

out = PROC / 'tableau_customer_detail.csv'
customer_detail.to_csv(out, index=False)
print(f"Saved: {out.name}  Shape: {customer_detail.shape}")
customer_detail.head()

Saved: tableau_customer_detail.csv  Shape: (5861, 15)


,customer_id,recency,frequency,monetary,R_score,F_score,M_score,segment,primary_country,total_transactions,unique_invoices,unique_products,first_purchase,last_purchase,customer_lifespan_days
0,12346,325,12,77556.46,2,5,5,At Risk,UNITED KINGDOM,34,12,27,2009-12-14 08:34:00,2011-01-18 10:01:00,400
1,12347,1,8,4921.53,5,4,5,Champions,ICELAND,222,8,126,2010-10-31 14:20:00,2011-12-07 15:52:00,402
2,12348,74,5,1658.40,3,4,4,Big Spenders,FINLAND,46,5,24,2010-09-27 14:59:00,2011-09-25 13:13:00,362
3,12349,18,3,3678.69,5,3,5,Big Spenders,ITALY,172,3,137,2010-04-29 13:20:00,2011-11-21 09:51:00,570
4,12350,309,1,294.40,2,1,2,Lost,NORWAY,16,1,16,2011-02-02 16:01:00,2011-02-02 16:01:00,0


## 6. Transactions with Segment Labels

In [7]:
trans_seg = trans.merge(
    rfm[['customer_id', 'segment']], on='customer_id', how='left'
)
trans_seg['segment'] = trans_seg['segment'].fillna('Unassigned')
trans_seg['month']   = trans_seg['invoice_date'].dt.to_period('M').astype(str)

out = PROC / 'tableau_transactions_with_segments.csv'
trans_seg.to_csv(out, index=False)
print(f"Saved: {out.name}  Shape: {trans_seg.shape}")
trans_seg.head()

Saved: tableau_transactions_with_segments.csv  Shape: (776641, 12)


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,revenue,year_month,segment,month
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,UNITED KINGDOM,83.4,2009-12,Big Spenders,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,UNITED KINGDOM,81.0,2009-12,Big Spenders,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,UNITED KINGDOM,81.0,2009-12,Big Spenders,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,UNITED KINGDOM,100.8,2009-12,Big Spenders,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,UNITED KINGDOM,30.0,2009-12,Big Spenders,2009-12


## 7. Project KPIs & File Manifest

In [8]:
print("=" * 60)
print("  OVERALL PROJECT KPIs")
print("=" * 60)
print(f"  Total Revenue      : £{rfm['monetary'].sum():>13,.0f}")
print(f"  Total Customers    : {len(rfm):>14,}")
print(f"  Avg Order Value    : £{trans['revenue'].mean():>13,.2f}")
print(f"  Unique Invoices    : {trans['invoice_no'].nunique():>14,}")
print(f"  Countries Served   : {trans['country'].nunique():>14,}")
print(f"  Date Range         : {trans['invoice_date'].min().date()} → {trans['invoice_date'].max().date()}")

print()
print("=" * 60)
print("  SEGMENT KPIs")
print("=" * 60)
print(segment_kpi[['segment','customers','pct_customers','total_revenue','pct_revenue']]
      .to_string(index=False))

print()
print("=" * 60)
print("  TABLEAU-READY FILES")
print("=" * 60)
for f in sorted(PROC.glob('tableau_*.csv')):
    row_count = sum(1 for _ in open(f)) - 1
    print(f"  {f.name:<47} {row_count:>9,} rows")
print()
print("All exports complete. Ready to import into Tableau Public.")

  OVERALL PROJECT KPIs
  Total Revenue      : £   17,073,078
  Total Customers    :          5,861
  Avg Order Value    : £        21.98
  Unique Invoices    :         36,639
  Countries Served   :             41
  Date Range         : 2009-12-01 → 2011-12-09

  SEGMENT KPIs
            segment  customers  pct_customers  total_revenue  pct_revenue
            At Risk       1530           26.1     1928758.50         11.3
       Big Spenders        683           11.7     2370959.86         13.9
          Champions       1477           25.2    11822980.86         69.2
               Lost        788           13.4      217628.03          1.3
    Loyal Customers        614           10.5      421945.80          2.5
Potential Loyalists        769           13.1      310805.24          1.8

  TABLEAU-READY FILES
  tableau_country_summary.csv                            41 rows
  tableau_customer_detail.csv                         5,861 rows
  tableau_monthly_revenue.csv                        